# 04 — Final RF/SVM decision, competition thresholds & deployment checks

**Purpose:** decide whether RF–Hybrid or SVM–Hybrid should be submitted. This notebook uses the *original TRAIN* (6,841) for threshold discovery, and the original *development validation* (1,074) for a limited check. It **never scores locked research test** (2,085). The separate hidden grader set is not accessible.

**Important:** development validation was previously inspected, so its score is *not* a pristine unbiased test. OOF threshold optimization is also selection-biased. Do not use the outcomes to repeatedly redesign the search. **No hyperparameter tuning is rerun here.**

Use in `/CDD203-HW1/`, where `hw1data.tab` lives. No matplotlib required.

In [1]:
from pathlib import Path
import time, json
import numpy as np
import pandas as pd
from IPython.display import display
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

SEED=42
OUT=Path('research_outputs/final_decision'); OUT.mkdir(parents=True,exist_ok=True)
DATA=Path('hw1data.tab')
assert DATA.exists(), 'Open notebook in CDD203-HW1'
RDLogger.DisableLog('rdApp.warning')

## 1. Recover the exact original partitions and hybrid features

In [2]:
df=pd.read_csv(DATA,sep='\t')
assert {'Drug','Y'}.issubset(df.columns)
smiles=df['Drug'].tolist()
mols=[Chem.MolFromSmiles(s) for s in smiles]
assert all(m is not None for m in mols)
y=df.Y.to_numpy(dtype=int)
groups=np.array([Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m),canonical=True) for m in mols])
idx=np.arange(len(df))
trainval_idx,test_idx=next(GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED).split(idx,y,groups))
a,b=next(GroupShuffleSplit(n_splits=1,test_size=.125,random_state=SEED).split(trainval_idx,y[trainval_idx],groups[trainval_idx]))
train_idx=trainval_idx[a]; val_idx=trainval_idx[b]
assert (len(train_idx),len(val_idx),len(test_idx))==(6841,1074,2085)
assert set(groups[train_idx]).isdisjoint(groups[val_idx])
assert set(groups[train_idx]).isdisjoint(groups[test_idx])

GEN=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048)
def hybrid_featurize(smiles_strings):
    output=np.zeros((len(smiles_strings),2058),dtype=np.float32)
    for i,s in enumerate(smiles_strings):
        m=Chem.MolFromSmiles(s)
        if m is None: raise ValueError(f'Invalid SMILES at row {i}: {s!r}')
        DataStructs.ConvertToNumpyArray(GEN.GetFingerprint(m),output[i,:2048])
        output[i,2048:]=[
            Descriptors.MolWt(m),Descriptors.MolLogP(m),rdMolDescriptors.CalcTPSA(m),
            rdMolDescriptors.CalcNumHBD(m),rdMolDescriptors.CalcNumHBA(m),
            rdMolDescriptors.CalcNumRotatableBonds(m),rdMolDescriptors.CalcNumAromaticRings(m),
            rdMolDescriptors.CalcFractionCSP3(m),Chem.GetFormalCharge(m),
            sum(atom.GetAtomicNum()==7 for atom in m.GetAtoms())]
    return output

X_train=hybrid_featurize([smiles[i] for i in train_idx])
X_val=hybrid_featurize([smiles[i] for i in val_idx])
y_train=y[train_idx]; y_val=y[val_idx]; g_train=groups[train_idx]
print('TRAIN:',X_train.shape,'DEV VALIDATION:',X_val.shape,'LOCKED TEST: NOT FEATURED OR SCORED')

TRAIN: (6841, 2058) DEV VALIDATION: (1074, 2058) LOCKED TEST: NOT FEATURED OR SCORED


## 2. Frozen settings from the completed focused tuning

In [3]:
RF=RandomForestClassifier(n_estimators=150,min_samples_leaf=1,max_features=0.2,
                          max_depth=30,class_weight='balanced',random_state=SEED,n_jobs=2)
SVM=Pipeline([
    ('pre',ColumnTransformer([('desc',StandardScaler(),list(range(2048,2058)))],remainder='passthrough')),
    ('model',SVC(C=3,gamma='scale',kernel='rbf',class_weight=None,probability=True,random_state=SEED))
])
CANDIDATES={'Random Forest':RF,'SVM-RBF':SVM}
print('Two candidates frozen; no additional search performed.')

Two candidates frozen; no additional search performed.


## 3. Generate scaffold-aware OOF predictions on original TRAIN (not on held-out data)

In [4]:
cv=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)
folds=list(cv.split(X_train,y_train,g_train))
assert all(not (set(g_train[tr]) & set(g_train[va])) for tr,va in folds)
oof={}; records=[]
for name,base in CANDIDATES.items():
    pred=np.full(len(y_train),np.nan)
    for fold,(tr,va) in enumerate(folds,1):
        est=clone(base)
        st=time.perf_counter(); est.fit(X_train[tr],y_train[tr]); elapsed=time.perf_counter()-st
        pred[va]=est.predict_proba(X_train[va])[:,list(est.classes_).index(1)]
        records.append({'Model':name,'Fold':fold,'train_seconds':elapsed})
    assert np.isfinite(pred).all()
    oof[name]=pred
    print(name,'OOF ROC-AUC',round(roc_auc_score(y_train,pred),4),
          'AP',round(average_precision_score(y_train,pred),4))
pd.DataFrame(records).to_csv(OUT/'training_fit_times.csv',index=False)

Random Forest OOF ROC-AUC 0.8544 AP 0.8667
SVM-RBF OOF ROC-AUC 0.8599 AP 0.8705


## 4. Choose thresholds with a robust, predefined rule

For each competition use *pooled OOF metric >= 0.75* **and** *worst-fold constraint > 0.70*. Within eligible thresholds, maximize the competition objective. This 0.75 buffer is a **research heuristic**, not a guaranteed hidden-test performance bound. We print both the pooled and worst-fold metrics to show feasibility. If no threshold qualifies, the corresponding entry is explicitly marked infeasible.

In [5]:
THRESHOLDS=np.round(np.arange(.01,1.00,.01),2)
def metrics(yt,p,t):
    z=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(yt,z,labels=[0,1]).ravel()
    sn=tp/(tp+fn) if tp+fn else np.nan
    sp=tn/(tn+fp) if tn+fp else np.nan
    pp=tp/(tp+fp) if tp+fp else 0.
    return {'Sensitivity':sn,'Specificity':sp,'Precision':pp,
            'Accuracy':(tp+tn)/len(yt), 'BalancedAccuracy':(sn+sp)/2,
            'TP':tp,'TN':tn,'FP':fp,'FN':fn}
all_rows=[]
for name,p in oof.items():
    for t in THRESHOLDS:
        q=metrics(y_train,p,t)
        fm=[metrics(y_train[va],p[va],t) for tr,va in folds]
        all_rows.append({'Model':name,'Threshold':t,**q,
                         'MinFoldSensitivity':min(v['Sensitivity'] for v in fm),
                         'MinFoldSpecificity':min(v['Specificity'] for v in fm),
                         'AUROC':roc_auc_score(y_train,p)})
curves=pd.DataFrame(all_rows)
curves.to_csv(OUT/'threshold_full_curves.csv',index=False)
COMPETITIONS={
 'Sensitivity':('Sensitivity','Specificity','MinFoldSpecificity'),
 'Specificity':('Specificity','Sensitivity','MinFoldSensitivity'),
 'Precision':('Precision','Sensitivity','MinFoldSensitivity')}
selected=[]
for name in CANDIDATES:
    d=curves[curves.Model==name]
    for title,(objective,constraint,worst) in COMPETITIONS.items():
        feasible=d[(d[constraint]>=.75)&(d[worst]>.70)]
        if feasible.empty:
            selected.append({'Model':name,'Competition':title,'Feasible':False})
            continue
        best=feasible.sort_values([objective,constraint],ascending=[False,False]).iloc[0]
        selected.append({'Model':name,'Competition':title,'Feasible':True,
                         'Threshold':best.Threshold,'Sensitivity':best.Sensitivity,
                         'Specificity':best.Specificity,'Precision':best.Precision,
                         'MinFoldSensitivity':best.MinFoldSensitivity,
                         'MinFoldSpecificity':best.MinFoldSpecificity})
threshold_choices=pd.DataFrame(selected)
display(threshold_choices.round(4))
threshold_choices.to_csv(OUT/'robust_threshold_choices.csv',index=False)

,Model,Competition,Feasible,Threshold,Sensitivity,Specificity,Precision,MinFoldSensitivity,MinFoldSpecificity
0,Random Forest,Sensitivity,True,0.49,0.7845,0.7624,0.7854,0.7647,0.7282
1,Random Forest,Specificity,True,0.51,0.7608,0.7861,0.7977,0.7454,0.7585
2,Random Forest,Precision,True,0.51,0.7608,0.7861,0.7977,0.7454,0.7585
3,SVM-RBF,Sensitivity,True,0.48,0.8009,0.7553,0.7839,0.7961,0.7317
4,SVM-RBF,Specificity,True,0.57,0.7531,0.8074,0.8125,0.7325,0.7799
5,SVM-RBF,Precision,True,0.57,0.7531,0.8074,0.8125,0.7325,0.7799


## 5. One limited development-validation check of the fixed candidates and thresholds

**Do not repeatedly adjust decisions based on this result.** The original development validation set was already examined in earlier research. We do not touch the locked research test.

In [6]:
validation_rows=[]
fitted_models={}
for name,base in CANDIDATES.items():
    fitted=clone(base).fit(X_train,y_train)
    fitted_models[name]=fitted
    p=fitted.predict_proba(X_val)[:,list(fitted.classes_).index(1)]
    for row in threshold_choices[threshold_choices.Model==name].itertuples():
        if not row.Feasible: continue
        z=metrics(y_val,p,row.Threshold)
        validation_rows.append({'Model':name,'Competition':row.Competition,
                                'Threshold':row.Threshold,'AUROC':roc_auc_score(y_val,p),
                                'AUPRC':average_precision_score(y_val,p),**z})
validation=pd.DataFrame(validation_rows)
display(validation.round(4))
validation.to_csv(OUT/'development_validation_check.csv',index=False)
print('WARNING: DEV VALIDATION is NOT pristine; do not overfit to these results.')

,Model,Competition,Threshold,AUROC,AUPRC,Sensitivity,Specificity,Precision,Accuracy,BalancedAccuracy,TP,TN,FP,FN
0,Random Forest,Sensitivity,0.49,0.8663,0.8742,0.7842,0.7683,0.7842,0.7765,0.7763,436,398,120,120
1,Random Forest,Specificity,0.51,0.8663,0.8742,0.7770,0.7876,0.7970,0.7821,0.7823,432,408,110,124
2,Random Forest,Precision,0.51,0.8663,0.8742,0.7770,0.7876,0.7970,0.7821,0.7823,432,408,110,124
3,SVM-RBF,Sensitivity,0.48,0.8860,0.8913,0.8201,0.7664,0.7903,0.7942,0.7933,456,397,121,100
4,SVM-RBF,Specificity,0.57,0.8860,0.8913,0.7626,0.8282,0.8265,0.7942,0.7954,424,429,89,132
5,SVM-RBF,Precision,0.57,0.8860,0.8913,0.7626,0.8282,0.8265,0.7942,0.7954,424,429,89,132


## 6. Measure featurization + inference wall time on CPU

Measured with repeated rounds on existing development-validation SMILES, not the locked test. This is a *local proxy* for grader ONNX throughput; the true competition requires ONNX on its hardware. **Neither training nor hyperparameter search time is the inference competition metric.**

In [8]:

# =====================================================
# 6. SPEED BENCHMARK — RF vs SVM
# Featurization + Prediction on CPU
# =====================================================

import time
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

# Use development validation SMILES, NOT locked test
SMILES_SAMPLE = [smiles[i] for i in val_idx]

N_REPEATS = 5
benchmark_results = []

for model_name, model in fitted_models.items():

    print(f"\nBenchmarking: {model_name}")

    # Warm-up (not timed)
    X_warm = hybrid_featurize(SMILES_SAMPLE[:10])
    model.predict_proba(X_warm)

    for repeat in range(N_REPEATS):

        with threadpool_limits(limits=2):

            # 1. Featurization time
            start = time.perf_counter()

            X_features = hybrid_featurize(SMILES_SAMPLE)

            feature_time = time.perf_counter() - start

            # 2. Prediction time
            start = time.perf_counter()

            probabilities = model.predict_proba(X_features)

            prediction_time = time.perf_counter() - start

        total_time = feature_time + prediction_time
        n = len(SMILES_SAMPLE)

        benchmark_results.append({
            "Model": model_name,
            "Repeat": repeat + 1,
            "Molecules": n,
            "Featurization_seconds": feature_time,
            "Prediction_seconds": prediction_time,
            "Total_seconds": total_time,
            "Molecules_per_second": n / total_time
        })

# Full results
benchmark_df = pd.DataFrame(benchmark_results)

# Summary across repeats
speed_summary = (
    benchmark_df
    .groupby("Model")
    [[
        "Featurization_seconds",
        "Prediction_seconds",
        "Total_seconds",
        "Molecules_per_second"
    ]]
    .median()
    .reset_index()
)

display(speed_summary.round(4))

# Save results
benchmark_df.to_csv(
    OUT / "speed_benchmark_all_repeats.csv",
    index=False
)

speed_summary.to_csv(
    OUT / "speed_benchmark_summary.csv",
    index=False
)

print("\nSpeed benchmark completed.")
print("Locked Test was NOT used.")



Benchmarking: Random Forest

Benchmarking: SVM-RBF


,Model,Featurization_seconds,Prediction_seconds,Total_seconds,Molecules_per_second
0,Random Forest,0.6733,0.0333,0.7061,1521.1188
1,SVM-RBF,0.6744,4.9707,5.6452,190.2511



Speed benchmark completed.
Locked Test was NOT used.


## Decision and submission handoff

- **Provisional submission model: RF–Hybrid**, because performance is close and it is simpler to deploy and benchmark. **SVM–Hybrid stays a comparator** until independent evidence and ONNX speed suggest otherwise.
- Threshold must be chosen deliberately from `robust_threshold_choices.csv` in light of the *single* competition target and safety rationale; the development-validation results are supporting evidence, not an invitation to repeatedly tune.
- A separately supplied `Part1_RF_HYBRID_DRAFT.py` implements exact same 2058-dimensional features, exports a fixed RF ONNX and supports varying SMILES lengths. **Set `NAME` and `THRESHOLD` before submission.**
- Verify `TestModel.py`, ONNX size ≤50 MB, CPU training <30 minutes, web app equivalence, `requirements.txt` and README. A grader-hidden test remains untouched by us.

In [9]:

# ============================================================
# 7. FINAL MODEL FREEZE + ONE-TIME LOCKED TEST
# ============================================================

import json
import time
from datetime import datetime, timezone
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score

# Prevent accidental repeat in the same kernel
assert not globals().get("LOCKED_TEST_EVALUATED", False), \
    "Locked Test already evaluated in this session."

# 1. FREEZE DECISIONS BEFORE OPENING LOCKED TEST
FINAL_MODEL_NAME = "RF-Hybrid"
FINAL_THRESHOLD = 0.49

final_model = clone(RF)

freeze_config = {
    "model": FINAL_MODEL_NAME,
    "threshold": FINAL_THRESHOLD,
    "representation": "Morgan radius=2, 2048 bits + 10 descriptors",
    "model_parameters": final_model.get_params(),
    "train_rows": int(len(train_idx) + len(val_idx)),
    "locked_test_rows": int(len(test_idx)),
    "frozen_at_utc": datetime.now(timezone.utc).isoformat(),
    "rule": "No changes after viewing locked-test results"
}

freeze_path = OUT / "final_model_freeze.json"

# Refuse to overwrite an existing freeze
assert not freeze_path.exists(), \
    "Freeze file already exists. Do not overwrite it."

with open(freeze_path, "x") as f:
    json.dump(freeze_config, f, indent=2, default=str)

print("MODEL FROZEN:", FINAL_MODEL_NAME)
print("THRESHOLD:", FINAL_THRESHOLD)
print("Freeze file:", freeze_path)

# 2. FIT ON TRAIN + DEVELOPMENT VALIDATION
development_idx = np.concatenate([train_idx, val_idx])

X_development = hybrid_featurize(
    [smiles[i] for i in development_idx]
)
y_development = y[development_idx]

start = time.perf_counter()
final_model.fit(X_development, y_development)
training_seconds = time.perf_counter() - start

print("\nFinal training complete")
print("Training molecules:", len(y_development))
print("Training seconds:", round(training_seconds, 2))

# 3. ONE-TIME LOCKED TEST EVALUATION
LOCKED_TEST_EVALUATED = True

X_locked = hybrid_featurize(
    [smiles[i] for i in test_idx]
)
y_locked = y[test_idx]

start = time.perf_counter()
p_locked = final_model.predict_proba(X_locked)[:, 1]
inference_seconds = time.perf_counter() - start

results = metrics(
    y_locked,
    p_locked,
    FINAL_THRESHOLD
)

results.update({
    "Model": FINAL_MODEL_NAME,
    "Threshold": FINAL_THRESHOLD,
    "AUROC": roc_auc_score(y_locked, p_locked),
    "AUPRC": average_precision_score(y_locked, p_locked),
    "TrainingSeconds": training_seconds,
    "PredictionSeconds": inference_seconds,
    "N_Test": len(y_locked)
})

final_results = pd.DataFrame([results])

print("\n========== LOCKED TEST RESULTS ==========")
display(final_results.round(4))

# 4. SAVE RESULTS
final_results.to_csv(
    OUT / "final_locked_test_results.csv",
    index=False
)

print("\nResults saved.")
print("Model and threshold must remain frozen.")


MODEL FROZEN: RF-Hybrid
THRESHOLD: 0.49
Freeze file: research_outputs/final_decision/final_model_freeze.json

Final training complete
Training molecules: 7915
Training seconds: 9.47

========== LOCKED TEST RESULTS ==========


,Sensitivity,Specificity,Precision,Accuracy,BalancedAccuracy,TP,TN,FP,FN,Model,Threshold,AUROC,AUPRC,TrainingSeconds,PredictionSeconds,N_Test
0,0.7915,0.7922,0.8049,0.7918,0.7919,858,793,208,226,RF-Hybrid,0.49,0.8806,0.899,9.4707,0.0342,2085



Results saved.
Model and threshold must remain frozen.
